# Demo: Ứng dụng tích phân trong máy bay (Đề tài 16 – Giải tích 2)

Notebook chạy lại toàn bộ bài từ đầu theo đúng thứ tự trong README (Bước 1 → 7), dùng các module trong `src/` và tham số trong `params.yaml`.
Chạy lần lượt từng ô (Run All). Kết quả bảng/hình được lưu vào `results/tables` và `results/figures`.

> Mô hình theo giả thiết của bài tập, không phải mô phỏng CFD.
<!-- AI-assisted: xem AI_DECLARATION.md -->

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src.config import load_params, FIGURES_DIR, TABLES_DIR, ensure_output_dirs
p = load_params(); ensure_output_dirs()
print("Thư mục dự án:", ROOT)
print(f"q_inf = {p.q_inf:.1f} Pa, alpha = {p.flow.alpha_deg}°, NACA {p.airfoil.naca}")

## Bước 1 — Airfoil NACA 4 chữ số (`airfoil.py`)

In [ ]:
from src.airfoil import parse_naca4, naca4_thickness, get_airfoil_coordinates
from src.visualize import plot_airfoil_2d
m, pp, t = parse_naca4(p.airfoil.naca)
print(f"m = {m}, p = {pp}, t = {t};  y_t(1) = {naca4_thickness(1.0, t):.2e} (mép sau khép kín)")
plot_airfoil_2d(p.airfoil.naca, FIGURES_DIR / "airfoil_profile.png")
display(Image(filename=str(FIGURES_DIR / "airfoil_profile.png"), width=700))

## Bước 2 — Cánh 3D và diện tích (tích phân kép, tích phân mặt) (`wing.py`)

In [ ]:
from src.wing import calculate_projected_area, calculate_surface_area
from src.visualize import plot_wing_planform_2d
w = p.wing
S_exact = calculate_projected_area(w.span, w.chord_root, w.chord_tip)
S_num = calculate_projected_area(w.span, w.chord_root, w.chord_tip, "numerical")
A = calculate_surface_area(w.span, w.chord_root, w.chord_tip, p.airfoil.naca, w.n_chord, w.n_span)
print(f"S (hình thang) = {S_exact:.4f} m², S (dblquad) = {S_num:.4f} m²")
print(f"Diện tích mặt trên/dưới/tổng = {A['area_upper']:.4f} / {A['area_lower']:.4f} / {A['total_surface_area']:.4f} m²")
plot_wing_planform_2d(p, FIGURES_DIR / "wing_planform_2d.png")
display(Image(filename=str(FIGURES_DIR / "wing_planform_2d.png"), width=700))

## Bước 3 — Thân máy bay: thể tích, khối lượng (tích phân bội ba), diện tích mặt (`fuselage.py`)

In [ ]:
from src.fuselage import (calculate_fuselage_volume, calculate_fuselage_mass,
                          analytical_fuselage_volume, calculate_fuselage_surface_area)
from src.visualize import plot_fuselage_profile_2d, plot_skin_mass_vs_thickness
f = p.fuselage; kw = dict(nose_length=f.nose_length, tail_length=f.tail_length)
v1 = calculate_fuselage_volume(f.length, f.radius_max, f.skin_thickness, **kw)
v3 = calculate_fuselage_volume(f.length, f.radius_max, f.skin_thickness, method="triple_integral", **kw)
print(f"V đặc: quad = {v1['volume_solid']:.6f}, tplquad = {v3['volume_solid']:.6f}, công thức = {analytical_fuselage_volume(f.length, f.radius_max, **kw):.6f} m³")
print(f"V vỏ = {v3['volume_skin']:.6f} m³, m vỏ = {calculate_fuselage_mass(f.length, f.radius_max, f.density, f.skin_thickness, True, **kw):.2f} kg")
print(f"Diện tích mặt thân = {calculate_fuselage_surface_area(f.length, f.radius_max, **kw):.4f} m²")
plot_fuselage_profile_2d(p, FIGURES_DIR / "fuselage_profile_2d.png")
plot_skin_mass_vs_thickness(p, FIGURES_DIR / "skin_mass_vs_thickness.png")
display(Image(filename=str(FIGURES_DIR / "fuselage_profile_2d.png"), width=700))
display(Image(filename=str(FIGURES_DIR / "skin_mass_vs_thickness.png"), width=550))

## Bước 4 — Lực nâng, lực cản bằng tích phân mặt (`aero.py`)

In [ ]:
from src.wing import WingModel
from src.aero import compute_aero_forces
from src.visualize import plot_cp_distribution, plot_span_loading
wing = WingModel(w.span, w.chord_root, w.chord_tip, p.airfoil.naca)
res = compute_aero_forces(wing, p)
res.print_table()
plot_cp_distribution(p, FIGURES_DIR / "cp_distribution.png", wing=wing)
plot_span_loading(p, FIGURES_DIR / "span_loading.png", wing=wing)
display(Image(filename=str(FIGURES_DIR / "cp_distribution.png"), width=600))
display(Image(filename=str(FIGURES_DIR / "span_loading.png"), width=550))

### Khảo sát theo góc tới và vận tốc

In [ ]:
from src.visualize import sweep_alpha, plot_lift_drag_polar, plot_lift_vs_velocity
sw = sweep_alpha(p, np.arange(-2, 13, 1.0), wing)
display(pd.DataFrame(sw).round(4))
plot_lift_drag_polar(sw["alpha"], sw["L"], sw["D"], FIGURES_DIR / "lift_drag_polar.png", q_S=p.q_inf * S_exact)
plot_lift_vs_velocity(p, FIGURES_DIR / "lift_vs_velocity.png", wing=wing)
display(Image(filename=str(FIGURES_DIR / "lift_drag_polar.png"), width=900))
display(Image(filename=str(FIGURES_DIR / "lift_vs_velocity.png"), width=550))

## Bước 5 — Kiểm chứng: số vs giải tích, hội tụ lưới (`validate.py`)

In [ ]:
from src.validate import validate_geometry, validate_aerodynamics, check_mesh_convergence, check_aero_convergence
from src.visualize import plot_convergence
display(validate_geometry(p)[["Đại lượng", "Tích phân số", "Tham chiếu", "Sai số tương đối (%)", "Đánh giá"]])
display(validate_aerodynamics(p))
cg, ca = check_mesh_convergence(p), check_aero_convergence(p)
plot_convergence(cg, ca, FIGURES_DIR / "convergence.png")
display(Image(filename=str(FIGURES_DIR / "convergence.png"), width=550))

## Bước 6 — Trực quan hóa 3D (`visualize.py`, phần dành cho Chương 5)

In [ ]:
from src.visualize import plot_wing_3d_matplotlib, plot_fuselage_3d_matplotlib, render_full_aircraft_plotly
plot_wing_3d_matplotlib(p, FIGURES_DIR / "wing_3d.png")
plot_fuselage_3d_matplotlib(p, FIGURES_DIR / "fuselage_3d.png")
render_full_aircraft_plotly(p, FIGURES_DIR / "aircraft_model.html")
display(Image(filename=str(FIGURES_DIR / "wing_3d.png"), width=650))
print("Mô hình tương tác:", FIGURES_DIR / "aircraft_model.html")

## Bước 7 — Chạy toàn bộ pipeline (`main.py`)

In [ ]:
from main import run_pipeline
summary = run_pipeline(p)
print({k: round(v, 4) for k, v in summary["geometry"].items()})
print(f"L = {summary['aerodynamics']['L']:.2f} N, D = {summary['aerodynamics']['D_total']:.2f} N")
print(sorted(x.name for x in FIGURES_DIR.iterdir()))